This notebook is the actual machine learning example, taking the pickle file from the train_biomass_model.ipynb notebook and applies it to actual satellite imagery from Google Earth Engine (GEE) to create a biomass map.

Here is how it works in plain English:

1. Connecting to the Source (GEE)
The script starts by reaching out to Google Earth Engine. It defines exactly what data it wants:

Where: A specific polygon (Area of Interest) in California.
When: All Landsat 8 images captured between 2015 and 2019.
What: The seven spectral bands (SR_B1 through SR_B7) that the model expects.

2. The Prediction UDF (predict_biomass)
This is the custom logic (UDF) that will run on every piece of the satellite map:

It receives a "chunk" of the satellite data as a table.
It checks for missing data (NaNs) so it doesn't waste time trying to predict "empty" pixels (like those obscured by clouds).
It feeds the valid pixels into your loaded_model to generate a biomass prediction.
It returns only the "Predicted Biomass" column.

3. Run the Computation!

robustraster:

Breaks the massive area into smaller 256x256 chunks.
It sets up the Dask workers to work on different chunks simulteneously.
It automatically sends your trained model to each worker so they all know how to make predictions.
As the workers finish, it gathers the results and stitches them back together into a GeoTIFF file.

In [ ]:
# 1. Import the Google Earth Engine (GEE) Python client library.
import ee

# Authenticate the current session with Google Earth Engine.
# This opens a browser-based OAuth2 flow to grant access.
# Authentication only needs to be performed once per machine.
ee.Authenticate()

In [1]:
# 2. Import the GEE library again (safe to re-import) and
#    initialize the Earth Engine session using the high-volume
#    endpoint, which is optimized for handling large numbers of
#    concurrent tile requests from distributed workflows.
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [2]:
# 3. Define a small Area of Interest (AOI) as a GEE Polygon geometry.
#    The coordinates describe a bounding box located in California's
#    Sierra Nevada region, specified in WGS84 (EPSG:4326) longitude/latitude.
#    The polygon is closed by repeating the first coordinate at the end.
aoi = ee.Geometry.Polygon(
    [[[-120.5, 38.5],     # Southwest corner (lower-left)
      [-120.5, 38.55],    # Northwest corner (upper-left)
      [-120.4, 38.55],    # Northeast corner (upper-right)
      [-120.4, 38.5],     # Southeast corner (lower-right)
      [-120.5, 38.5]]]    # Close the polygon at the starting vertex
)

In [4]:
# 4. Define the temporal range for the Landsat 8 image collection.
#    A 5-year window provides a multi-temporal stack of imagery
#    from which biomass predictions will be generated.
start_date = '2015-01-01'  # Start of the observation period
end_date = '2019-12-31'    # End of the observation period

def prep_sr_l8(image):
    """Applies Landsat 8 Collection 2 Level-2 scaling factors to convert
    raw digital numbers into physical surface reflectance values."""

    # Select all surface reflectance bands (matching the 'SR_B' prefix)
    # and apply the Collection 2 Level-2 scaling formula:
    #   Reflectance = (DN × 0.0000275) + (−0.2)
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)

    # Replace the original unscaled bands with the newly scaled bands.
    # The third argument (True) enables overwriting existing band names.
    return image.addBands(optical_bands, None, True)

# Build the Landsat 8 Image Collection by:
#   1. Loading the USGS Landsat 8 Collection 2 Tier 1 Level-2 product
#   2. Filtering to only images that intersect the AOI
#   3. Filtering to the defined 5-year date range
#   4. Applying the surface reflectance scaling function to each image
#   5. Selecting all 7 surface reflectance bands required by the ML model:
#      Coastal Aerosol (B1), Blue (B2), Green (B3), Red (B4),
#      NIR (B5), SWIR-1 (B6), SWIR-2 (B7)
ic = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(aoi)
      .filterDate(start_date, end_date)
      .map(prep_sr_l8)
      .select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']))

In [5]:
# 5. Import the libraries needed by the user-defined function (UDF)
#    and define the custom prediction function.

# Import Pandas for DataFrame operations within the UDF
import pandas as pd
# Import NumPy for numerical computations and NaN handling
import numpy as np
# Import joblib for deserializing the pre-trained model from disk
import joblib
# Import os for filesystem path checks
import os

def predict_biomass(df, ml_model):
    """
    Takes a chunk of Landsat 8 data (as a pandas DataFrame) and a
    pre-trained scikit-learn model. Predicts biomass for each valid
    pixel observation using all 7 surface reflectance bands as features.

    Parameters:
        df       : pandas DataFrame containing the spectral band columns
                    and spatial/temporal metadata supplied by robustraster.
        ml_model : A pre-trained scikit-learn model object (e.g.,
                    GradientBoostingRegressor) passed via user_function_kwargs.

    Returns:
        A DataFrame with a single 'predicted_biomass' column, having
        the same number of rows as the input for spatial remapping.
    """

    # Create a copy of the input DataFrame to avoid modifying the original
    df = df.copy()

    # Define the 7 Landsat 8 surface reflectance band names that
    # correspond to the features the model was trained on
    bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']

    # Build a boolean mask identifying rows where ALL 7 bands contain
    # valid (non-NaN) data. Predictions require complete feature vectors.
    valid_mask = df[bands].notna().all(axis=1)

    # Initialize the output column with NaN so that pixels lacking
    # valid data are preserved as NoData in the final raster
    df['predicted_biomass'] = np.nan

    # Only run the model if at least one row has valid data in this chunk
    if valid_mask.any():
        # Extract the feature matrix (X) for valid rows only,
        # shaped as (n_valid_pixels, 7)
        X = df.loc[valid_mask, bands]

        # Generate biomass predictions using the pre-trained model.
        # The model's .predict() method returns a 1D NumPy array
        # of predicted biomass values (one per valid pixel).
        predictions = ml_model.predict(X)

        # Assign the predicted values back to their corresponding
        # positions in the full-sized DataFrame
        df.loc[valid_mask, 'predicted_biomass'] = predictions

    # Return only the output column. robustraster uses this to
    # construct the output raster bands.
    return df[['predicted_biomass']]

In [ ]:
# 6. Load the pre-trained Gradient Boosting model that was previously
#    generated and exported by train_biomass_model.ipynb.

# Define the path to the serialized model file
model_path = "biomass_model.pkl"

# Verify that the model file exists before attempting to load it.
# If missing, raise a descriptive error directing the user to
# run the training notebook first.
if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Could not find '{model_path}'. "
        "Please run the code from 'train_biomass_model.ipynb' first to generate the random model."
    )

# Log that the model is being loaded
print(f"Loading pre-trained model from {model_path}...")

# Deserialize the trained model object from the pickle file using joblib.
# Dask will automatically serialize this object and distribute it to
# all worker processes when the UDF is executed in parallel.
loaded_model = joblib.load(model_path)

In [ ]:
# 7. Execute the full computation pipeline using robustraster.

# Import the main run() entry point from the robustraster package
from robustraster import run

# Log the start of the ML prediction workflow
print("Starting Machine Learning prediction on Landsat imagery...")

# Define the chunking strategy for Dask-backed xarray processing.
# "time": -1 loads ALL time steps into memory for each spatial tile.
#   While not strictly required for point-in-time predictions, this
#   maintains consistency with other demos and avoids partial loads.
# "X": 256 and "Y": 256 set the spatial tile size to 256×256 pixels,
#   balancing memory usage with computational efficiency.
chunks = {"time": -1, "X": 256, "Y": 256}

# Launch the robustraster processing pipeline with all configuration:
run(
    # The Earth Engine ImageCollection to process
    dataset=ic,

    # Dataset configuration: spatial/projection parameters
    dataset_config={
        'geometry': aoi,        # The AOI polygon defining the spatial extent
        'crs': 'EPSG:3310',     # California Albers Equal Area projection
        'scale': 30,            # Landsat 8 native spatial resolution (30 m)
    },

    # User function configuration: the prediction UDF and its arguments
    user_function_config={
        "user_function": predict_biomass,   # The function applied to each chunk
        "user_function_args": (),           # No positional arguments needed
        # Pass the deserialized model object as a keyword argument.
        # robustraster forwards this to each invocation of predict_biomass().
        "user_function_kwargs": {"ml_model": loaded_model},
    },

    # Performance tuning: Dask chunk dimensions
    function_tuning_config={
        "chunks": chunks,  # Chunking strategy defined above
    },

    # Export configuration: output format and destination
    export_config={
        "mode": "raster",                       # Export results as a raster file
        "file_format": "GTiff",                  # GeoTIFF output format
        "output_folder": "ML_Prediction_Output", # Directory for output files
        "vrt": True,                             # Generate a GDAL Virtual Raster mosaic
        "report": True                           # Generate a Dask performance report
    }
)
